In [3]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# 1. Find the project root
ROOT = Path.cwd().resolve()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

# 2. Import our reusable HOG extraction function
sys.path.insert(0, str(ROOT))

from src.features import extract_hog_dataset

# 3. Load existing driver-based training and validation splits
train_df = pd.read_csv(
    ROOT / "data/splits/driver_based/train.csv"
)

val_df = pd.read_csv(
    ROOT / "data/splits/driver_based/val.csv"
)

# 4. Verify training and validation are separate
assert set(train_df["img"]).isdisjoint(set(val_df["img"]))
assert set(train_df["subject"]).isdisjoint(set(val_df["subject"]))

print("Training images:", len(train_df))
print("Validation images:", len(val_df))

# 5. Define where images and cached features are stored
image_dir = ROOT / "data/raw/imgs/train"
cache_dir = ROOT / "data/processed/driver_based"

cache_dir.mkdir(parents=True, exist_ok=True)

# 6. Define HOG image resolutions (width, height)
resolutions = {
    "64x48": (64, 48),
    "128x96": (128, 96),
    "256x192": (256, 192)
}

# 7. Extract and save HOG features for training and validation
for split_name, split_df in [
    ("train", train_df),
    ("val", val_df)
]:
    for resolution_name, image_size in resolutions.items():

        feature_path = cache_dir / f"hog_{resolution_name}_{split_name}.npy"

        # Load existing features instead of extracting again
        if feature_path.exists():
            saved_X = np.load(feature_path, mmap_mode="r")

            if saved_X.shape[0] != len(split_df):
                raise ValueError(f"Unexpected number of rows in {feature_path}")

            print(f"Already saved: {feature_path.name} {saved_X.shape}")
            del saved_X
            continue

        print(f"\nExtracting {split_name} HOG features at {resolution_name}...")

        # Extract HOG features from images
        X = extract_hog_dataset(
            split_df,
            image_dir,
            image_size=image_size
        )

        # Save features permanently
        np.save(feature_path, X)

        print(f"Saved: {feature_path.name}, Shape: {X.shape}")

        # Free memory before processing the next resolution
        del X

print("\nAll training and validation HOG features are ready!")

Training images: 15732
Validation images: 3603

Extracting train HOG features at 64x48...


  0%|          | 0/15732 [00:00<?, ?it/s]

100%|██████████| 15732/15732 [00:11<00:00, 1347.38it/s]


Saved: hog_64x48_train.npy, Shape: (15732, 1260)

Extracting train HOG features at 128x96...


100%|██████████| 15732/15732 [00:26<00:00, 596.70it/s]


Saved: hog_128x96_train.npy, Shape: (15732, 5940)

Extracting train HOG features at 256x192...


... (4 outputs left out)

100%|██████████| 3603/3603 [00:06<00:00, 591.95it/s]


Saved: hog_128x96_val.npy, Shape: (3603, 5940)

Extracting val HOG features at 256x192...


100%|██████████| 3603/3603 [00:20<00:00, 171.88it/s]

Saved: hog_256x192_val.npy, Shape: (3603, 25668)

All training and validation HOG features are ready!
